# PART 1 — Data Preprocessing

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("tmdb_5000_movies.csv")

print("Dataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
print(df.head())

print("\nDataset Information:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape:
(4803, 20)

Column Names:
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']

First 5 Rows:
      budget                                             genres  \
0  237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  250000000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  260000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                       homepage      id  \
0                   http://www.avatarmovie.com/   19995   
1  http://disney.go.com/disneypictures/pirates/     285   
2   http://www.sonypictures.com/movies/spectre/  206647   
3            http://www

In [6]:
text_column = "overview"
item_column = "title"

print("Item column:", item_column)
print("Text column:", text_column)

Item column: title
Text column: overview


# Task 2: Text Preprocessing

In [7]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Replace missing values
df["overview"] = df["overview"].fillna("")

def clean_text(text):
    
    text = str(text).lower()
    
    # Keep only letters and spaces
    text = "".join(
        char if char.isalpha() or char.isspace() else " "
        for char in text
    )
    
    # Split into words
    words = text.split()
    
    # Remove stopwords
    words = [
        word for word in words
        if word not in ENGLISH_STOP_WORDS
    ]
    
    return " ".join(words)


df["clean_text"] = df["overview"].apply(clean_text)

print(df[["title", "overview", "clean_text"]].head(10))

                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   
5                              Spider-Man 3   
6                                   Tangled   
7                   Avengers: Age of Ultron   
8    Harry Potter and the Half-Blood Prince   
9        Batman v Superman: Dawn of Justice   

                                            overview  \
0  In the 22nd century, a paraplegic Marine is di...   
1  Captain Barbossa, long believed to be dead, ha...   
2  A cryptic message from Bond’s past sends him o...   
3  Following the death of District Attorney Harve...   
4  John Carter is a war-weary, former military ca...   
5  The seemingly invincible Spider-Man goes up ag...   
6  When the kingdom's most wanted-and most charmi...   
7  When Tony Stark tries to jumpst

In [8]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Replace missing values
df["overview"] = df["overview"].fillna("")

def clean_text(text):
    
    text = str(text).lower()
    
    # Keep only letters and spaces
    text = "".join(
        char if char.isalpha() or char.isspace() else " "
        for char in text
    )
    
    # Split into words
    words = text.split()
    
    # Remove stopwords
    words = [
        word for word in words
        if word not in ENGLISH_STOP_WORDS
    ]
    
    return " ".join(words)


df["clean_text"] = df["overview"].apply(clean_text)

print(df[["title", "overview", "clean_text"]].head(10))

                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   
5                              Spider-Man 3   
6                                   Tangled   
7                   Avengers: Age of Ultron   
8    Harry Potter and the Half-Blood Prince   
9        Batman v Superman: Dawn of Justice   

                                            overview  \
0  In the 22nd century, a paraplegic Marine is di...   
1  Captain Barbossa, long believed to be dead, ha...   
2  A cryptic message from Bond’s past sends him o...   
3  Following the death of District Attorney Harve...   
4  John Carter is a war-weary, former military ca...   
5  The seemingly invincible Spider-Man goes up ag...   
6  When the kingdom's most wanted-and most charmi...   
7  When Tony Stark tries to jumpst

In [9]:
print("Missing clean text values:")
print(df["clean_text"].isnull().sum())

Missing clean text values:
0


# PART 2 — Text Vectorization
# Task 3: TF-IDF Vectorization

TF-IDF Matrix Shape:
(4803, 5000)

Number of Features:
5000

Sample Features:
['aaron' 'abandoned' 'abandons' 'abby' 'abducted' 'abilities' 'ability'
 'able' 'aboard' 'abroad' 'absence' 'abuse' 'abused' 'abusive' 'academic'
 'academy' 'accept' 'accepted' 'accepts' 'access']


# Task 4: Similarity Computation

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity Matrix Shape:")
print(similarity_matrix.shape)

Similarity Matrix Shape:
(4803, 4803)


# Why cosine similarity?

Cosine similarity measures how similar two text vectors are by comparing the angle between them. It is useful for text recommendation because it focuses on the direction of the TF-IDF vectors rather than their magnitude. Movies with similar descriptions will generally have higher cosine similarity scores.

# PART 3 — Recommendation Logic
# Task 5: Build Recommendation Function

In [13]:
def recommend(item_name, top_n=5):
    
    # Find the index of the selected movie
    matches = df.index[
        df["title"].str.lower() == item_name.lower()
    ].tolist()
    
    if len(matches) == 0:
        return "Movie not found."
    
    item_index = matches[0]
    
    # Get similarity scores for selected movie
    similarity_scores = list(
        enumerate(similarity_matrix[item_index])
    )
    
    # Sort by similarity score
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )
    
    # Remove the selected movie itself
    similarity_scores = [
        score for score in similarity_scores
        if score[0] != item_index
    ]
    
    # Get top N recommendations
    top_items = similarity_scores[:top_n]
    
    recommendations = []
    
    for index, score in top_items:
        recommendations.append({
            "title": df.iloc[index]["title"],
            "similarity": round(score, 4)
        })
    
    return pd.DataFrame(recommendations)

In [14]:
print("Recommendation 1:")
print(recommend("Avatar", 5))

print("\nRecommendation 2:")
print(recommend("Titanic", 5))

print("\nRecommendation 3:")
print(recommend("The Dark Knight", 5))

Recommendation 1:
                  title  similarity
0             Apollo 18      0.2282
1          The American      0.1981
2      Tears of the Sun      0.1519
3               Beowulf      0.1495
4  The Inhabited Island      0.1490

Recommendation 2:
                        title  similarity
0                  The Switch      0.2269
1    Amidst the Devil's Wings      0.2107
2                 End of Days      0.1885
3                  Ghost Ship      0.1883
4  I Can Do Bad All By Myself      0.1683

Recommendation 3:
                                     title  similarity
0                    The Dark Knight Rises      0.3724
1                           Batman Forever      0.3156
2                           Batman Returns      0.2753
3  Batman: The Dark Knight Returns, Part 2      0.2447
4                                Slow Burn      0.2212


In [15]:
print(df["title"].head(20).tolist())

['Avatar', "Pirates of the Caribbean: At World's End", 'Spectre', 'The Dark Knight Rises', 'John Carter', 'Spider-Man 3', 'Tangled', 'Avengers: Age of Ultron', 'Harry Potter and the Half-Blood Prince', 'Batman v Superman: Dawn of Justice', 'Superman Returns', 'Quantum of Solace', "Pirates of the Caribbean: Dead Man's Chest", 'The Lone Ranger', 'Man of Steel', 'The Chronicles of Narnia: Prince Caspian', 'The Avengers', 'Pirates of the Caribbean: On Stranger Tides', 'Men in Black 3', 'The Hobbit: The Battle of the Five Armies']


# Task 10